In [1]:
from src.data import DatasetLoader, GraphBuilder, Preprocessor
from src.models import Features

dataset_loader = DatasetLoader()
data = dataset_loader.load_dataset()

preprocessor = Preprocessor(data)
processed_df = preprocessor.preprocess()

graph_builder = GraphBuilder(processed_df, node_features=Features(
        tweet=[
            "favorite_count",
            # "retweet_count",
            # "bookmark_count",
            "reply_count",
            "quote_count",
            # "views",
            "source",
            "is_hateful",
        ],
        user=[
            # "favourites_count",
            # "follower_count",
            # "following_count",
            # "number_of_tweets",
            # "listed_count",
            # "is_blue_verified",
            "friends",
        ],
    )
)

G = graph_builder.create_graph(heterogeneous=True)

2025-07-27 12:16:55.416 | INFO     | src.config:<module>:27 - Loaded environment variables from /home/iragca/Documents/github/capstone-project-2/.env
2025-07-27 12:16:55.417 | INFO     | src.config:<module>:64 - PROJECT_ROOT: /home/iragca/Documents/github/capstone-project-2
2025-07-27 12:16:55.417 | INFO     | src.config:<module>:65 - DATA_DIR: /home/iragca/Documents/github/capstone-project-2/data


✅ Loading cached dataset - Done.
✅ Loading dataset - Done.
✅ Categorizing source column - Done.
✅ Preprocessing data - Done.


In [2]:
G.number_of_edges(), G.number_of_nodes()

(35917, 60893)

In [3]:
for node in G.nodes(data=True):
    print(node)
    break

(1269916756719685632, {'node_feature': tensor([0., 0., 0., 0., 1.]), 'node_type': 'test_node_type'})


In [4]:
for edge in G.edges(data=True):
    e = edge
    print(edge)
    break

# G[e[0]][e[1]]['edge_type']

(1269916756719685632, 1264846057906806786, {'edge_feature': tensor([0., 0., 0., 0., 0.]), 'edge_type': 'replied_to'})


In [5]:
from deepsnap.hetero_graph import HeteroGraph

hetero2 = HeteroGraph(G)
for edge in G.edges(data=True):
    print(edge)
    break
print("Node types: {}".format(hetero2.node_types))
print("Edge types: {}".format(hetero2.edge_types))
print("Message types: {}".format(hetero2.message_types))
for node_type in hetero2.node_types:
    print("Node type {} has {} nodes".format(node_type, hetero2.num_nodes(node_type)))
for message_type in hetero2.message_types:
    print("Message type {} has {} edges".format(message_type, hetero2.num_edges(message_type)))
print(hetero2.node_feature)
print(hetero2.node_label)

(1269916756719685632, 1264846057906806786, {'edge_feature': tensor([0., 0., 0., 0., 0.]), 'edge_type': 'replied_to'})
Node types: ['test_node_type']
Edge types: ['replied_to']
Message types: [('test_node_type', 'replied_to', 'test_node_type')]
Node type test_node_type has 60893 nodes
Message type ('test_node_type', 'replied_to', 'test_node_type') has 35917 edges
{'test_node_type': tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [6.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.8600e+03, 3.3400e+02, 2.8000e+01, 0.0000e+00, 1.0000e+00],
        ...,
        [6.4000e+01, 0.0000e+00, 1.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [1.8000e+01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00]])}
None


In [6]:
# Functions to generate two internal GNN layers for link prediction task
def generate_2convs_link_pred_layers(hete, conv, hidden_size):
    convs1 = {}
    convs2 = {}
    for message_type in hete.message_types:
        n_type = message_type[0]
        s_type = message_type[2]
        n_feat_dim = hete.num_node_features(n_type)
        s_feat_dim = hete.num_node_features(s_type)
        convs1[message_type] = conv(n_feat_dim, hidden_size, s_feat_dim)
        convs2[message_type] = conv(hidden_size, hidden_size, hidden_size)
    return convs1, convs2

In [7]:
from pprint import pprint
from deepsnap.hetero_graph import HeteroGraph
from deepsnap.dataset import GraphDataset
from deepsnap.batch import Batch
from deepsnap.hetero_gnn import HeteroSAGEConv
from torch.utils.data import DataLoader

hetero = HeteroGraph(G)
hidden_size = 32

# Generate two heterogeneous GNN layers for link prediction
conv1, conv2 = generate_2convs_link_pred_layers(hetero, HeteroSAGEConv, hidden_size)
pprint(conv1)
pprint(conv2)

dataset = GraphDataset([hetero], task='link_pred')
dataset_train, dataset_val, dataset_test = dataset.split(transductive=True,
                                                        split_ratio=[0.8, 0.1, 0.1])
train_loader = DataLoader(dataset_train, collate_fn=Batch.collate(),
                    batch_size=1)
val_loader = DataLoader(dataset_val, collate_fn=Batch.collate(),
                    batch_size=1)
test_loader = DataLoader(dataset_test, collate_fn=Batch.collate(),
                    batch_size=1)
dataloaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}

{('test_node_type', 'replied_to', 'test_node_type'): HeteroSAGEConv(neigh: 5, self: 5, out: 32)}
{('test_node_type', 'replied_to', 'test_node_type'): HeteroSAGEConv(neigh: 32, self: 32, out: 32)}


In [8]:
import torch
import torch.nn as nn
from deepsnap.hetero_gnn import forward_op, HeteroConv
import numpy as np
import copy
# Define the heterogeneous GNN for the link prediction task
class HeteroGNN(torch.nn.Module):
    def __init__(self, conv1, conv2, hetero, hidden_size):
        super(HeteroGNN, self).__init__()

        self.convs1 = HeteroConv(conv1) # Wrap the heterogeneous GNN layers
        self.convs2 = HeteroConv(conv2)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()
        self.bns1 = nn.ModuleDict()
        self.bns2 = nn.ModuleDict()
        self.relus1 = nn.ModuleDict()
        self.relus2 = nn.ModuleDict()
        self.post_mps = nn.ModuleDict()

        for node_type in hetero.node_types:
            self.bns1[node_type] = torch.nn.BatchNorm1d(hidden_size)
            self.bns2[node_type] = torch.nn.BatchNorm1d(hidden_size)
            self.relus1[node_type] = nn.LeakyReLU()
            self.relus2[node_type] = nn.LeakyReLU()

    def forward(self, data):
        x = data.node_feature
        edge_index = data.edge_index
        x = self.convs1(x, edge_index)
        x = forward_op(x, self.bns1)
        x = forward_op(x, self.relus1)
        x = self.convs2(x, edge_index)
        x = forward_op(x, self.bns2)

        pred = {}
        for message_type in data.edge_label_index:
            nodes_first = torch.index_select(x['test_node_type'], 0, data.edge_label_index[message_type][0,:].long())
            nodes_second = torch.index_select(x['test_node_type'], 0, data.edge_label_index[message_type][1,:].long())
            pred[message_type] = torch.sum(nodes_first * nodes_second, dim=-1)
        return pred

    def loss(self, pred, y):
        loss = 0
        for key in pred:
            p = torch.sigmoid(pred[key])
            loss += self.loss_fn(p, y[key].type(pred[key].dtype))
        return loss

In [9]:


# Train function
def train(model, dataloaders, optimizer, args):
    val_max = 0
    best_model = model
    t_accu = []
    v_accu = []
    e_accu = []
    for epoch in range(1, args["epochs"] + 1):
        for iter_i, batch in enumerate(dataloaders['train']):
            batch.to(args["device"])
            model.train()
            optimizer.zero_grad()
            pred = model(batch)
            loss = model.loss(pred, batch.edge_label)
            loss.backward()
            optimizer.step()

            log = 'Epoch: {:03d}, Train loss: {:.4f}, Train: {:.4f}, Val: {:.4f}, Test: {:.4f}'
            accs = test(model, dataloaders, args)
            t_accu.append(accs['train'])
            v_accu.append(accs['val'])
            e_accu.append(accs['test'])

            print(log.format(epoch, loss.item(), accs['train'], accs['val'], accs['test']))
            if val_max < accs['val']:
                val_max = accs['val']
                best_model = copy.deepcopy(model)

    log = 'Best: Train: {:.4f}, Val: {:.4f}, Test: {:.4f}'
    accs = test(best_model, dataloaders, args)
    print(log.format(accs['train'], accs['val'], accs['test']))

    return t_accu, v_accu, e_accu

# Test function
def test(model, dataloaders, args):
    model.eval()
    accs = {}
    for mode, dataloader in dataloaders.items():
        acc = 0
        for i, batch in enumerate(dataloader):
            num = 0
            batch.to(args["device"])
            pred = model(batch)
            for key in pred:
                p = torch.sigmoid(pred[key]).cpu().detach().numpy()
                pred_label = np.zeros_like(p, dtype=np.int64)
                pred_label[np.where(p > 0.5)[0]] = 1
                pred_label[np.where(p <= 0.5)[0]] = 0
                acc += np.sum(pred_label == batch.edge_label[key].cpu().numpy())
                num += len(pred_label)
        accs[mode] = acc / num
    return accs

In [10]:
args = {
    "device": "cpu",
    "epochs": 50,
    "lr": 0.01,
    "weight_decay": 1e-4
}

# Build the model and start training
model = HeteroGNN(conv1, conv2, hetero, hidden_size).to(args["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'], weight_decay=args['weight_decay'])
t_accu, v_accu, e_accu = train(model, dataloaders, optimizer, args)

Epoch: 001, Train loss: 0.7307, Train: 0.5012, Val: 0.4829, Test: 0.4860
Epoch: 002, Train loss: 0.7101, Train: 0.5095, Val: 0.5001, Test: 0.5049
Epoch: 003, Train loss: 0.6997, Train: 0.5503, Val: 0.5192, Test: 0.5335
Epoch: 004, Train loss: 0.6935, Train: 0.6254, Val: 0.4869, Test: 0.5013
Epoch: 005, Train loss: 0.6840, Train: 0.6826, Val: 0.4297, Test: 0.4322
Epoch: 006, Train loss: 0.6785, Train: 0.7024, Val: 0.4373, Test: 0.4411
Epoch: 007, Train loss: 0.6747, Train: 0.7093, Val: 0.4476, Test: 0.4529
Epoch: 008, Train loss: 0.6727, Train: 0.7162, Val: 0.4648, Test: 0.4718
Epoch: 009, Train loss: 0.6703, Train: 0.7201, Val: 0.4694, Test: 0.4773
Epoch: 010, Train loss: 0.6690, Train: 0.7228, Val: 0.4816, Test: 0.4905
Epoch: 011, Train loss: 0.6676, Train: 0.7212, Val: 0.4999, Test: 0.5087
Epoch: 012, Train loss: 0.6666, Train: 0.7235, Val: 0.5103, Test: 0.5206
Epoch: 013, Train loss: 0.6650, Train: 0.7242, Val: 0.5187, Test: 0.5319
Epoch: 014, Train loss: 0.6640, Train: 0.7223, Val: